# 04 · Evaluate: the test set, opened once

*For learning. Nothing here runs in CI and nothing leaves this computer: the datasets registered by stage 1 on the local
MLflow server (`http://localhost:5050`, experiment `coach-retrieval-lab`), the fine-tuned candidate read back from MLflow,
`bge-large` from the Hugging Face cache, and Ollama for `embeddinggemma`.*

**What this notebook settles.** Three **contenders** (models competing for the same job) answer the same **test** questions:

| contender | what it is | role |
|---|---|---|
| `bge-large · local · with prompt` | bge-large exactly as its makers ship it, with its search prompt | **the champion**: what runs today, pay-per-token on Databricks |
| `bge-large · fine-tuned · ...` | stage 3's candidate: the run tagged `candidate=stage4`, loaded from its **logged MLflow model** | challenger (a) |
| `embeddinggemma 768 · ollama · with prompt` | embeddinggemma as its makers ship it (its weights are gated, so it cannot be fine-tuned here) | challenger (b) |

A challenger replaces the champion only if it clears **the bar**, and the bar is written down, logged and timed **before
the first test question is turned into a vector**. That order is the point of the notebook, so the sections follow it:

0. **Setup.** The frozen data, checked by digest; the honest test checked as read; the candidate loaded from MLflow and
   shown to be the model that was chosen.
1. **The bar**, written before the test is opened (pre-registration).
2. **Your guesses**, before any test number.
3. **Open the test**: the three contenders scored, once.
4. **Paired comparisons** against the champion, then the bar applied mechanically, in code.
5. **The decision**, with what would make it wrong, and dev against test.
6. **What stage 5 does next**, and why the holdout stays locked until then.

**The once-only rule.** The 29 **test** lessons were kept closed through stages 2 and 3, so they are the only questions
nothing was chosen on. They are opened here, once. **Re-running this notebook re-scores the same frozen models on the same
questions with the same bar, so it is the same look, not a new one** (the scores come out the same; the timing is not
measured again: the first look's speed passes are read back, so no verdict can change). What is not allowed, ever, after this notebook has run on test: **retraining** the candidate, **re-choosing**
a different run or checkpoint, or **changing the bar**. Any of those would turn test into a second dev set, and there would
be no clean number left. The code enforces it: section 0 refuses a second pass whose candidate differs from the first, and section 1 one whose bar
differs; section 5 refuses one whose decision differs and sets no second winner tag. Any pass that logged the moment it
opened test counts as a look, even one that crashed or was stopped afterwards.

**Who read the honest test.** The learner-worded test questions were written by a model (`qwen3:14b`) and read, at the
owner's request, by **Claude, a model, not a person**: each rewrite was compared with its original for the same meaning and
the same lesson, and the ones that changed the meaning were dropped. Section 0 checks every one has been read before any
test score exists. Every honest number below carries that caveat.

**The lessons applied.** Where the course has a lesson for a habit, its title is in the heading: *Dev, test and holdout
sets*, *Testing many variants*, *Measuring retrieval*, *Margins near 0% and 100%*, *Comparing on the same examples*,
*Telling real gains from noise*, *Reading regressions and improvements*, *Latency that users feel*, *Embedding models have
blind spots*, *Model registry and promotion*.

**How to work through it.** Before each result there is a cell for **your guess**: write it down, then run the next cell.
Your guesses go into the MLflow run beside the real answers. The exercises have blanks (`"___"`); the answers are in
`04_evaluate_answers.py`, which `DRILL_FILL_ANSWERS=1` uses to check the notebook runs end to end.

**How long it takes.** About five minutes: each contender embeds the course (744 paragraphs, about 10 s) and three
sets of test questions, then one untimed warm-up pass and five timed passes each.

> **In plain words: a dry run first.** Because the test is opened once, a crash half-way through would leave a half-done
> look. So the notebook can be run first on the **dev** lessons instead (`DRILL_DRY_RUN_ON_DEV=1`): every cell runs, against
> questions that were already used for choosing, logged as a run named "dry run · evaluate on dev (not a result)" and tagged
> so nothing mistakes it for the test look. The real term is a **dry run**: the whole procedure on stand-in data, to check
> the procedure, not the result (the glossary's *smoke test* is the nearest entry: a quick run to check every piece works
> together, whose numbers decide nothing). The dry run opens nothing and tags no winner.

In [ ]:
import hashlib
import importlib
import json
import os
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT", "1")   # run links are printed short, by tracking.run_link
os.environ.setdefault("TQDM_DISABLE", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # before any import that loads the Hugging Face library: the champion is in the cache, nothing is fetched

import mlflow
import pandas as pd
from drill.content import build_data
from mlflow.entities import LoggedModelStatus, ViewType

import retrieval as r
import tracking
import training as t

FILL = os.environ.get("DRILL_FILL_ANSWERS") == "1"          # only for checking the notebook end to end
DRY_RUN = os.environ.get("DRILL_DRY_RUN_ON_DEV") == "1"     # every cell on dev instead of test: checks the code, opens nothing
SPLIT = "dev" if DRY_RUN else "test"
RUN_NAME = "dry run · evaluate on dev (not a result)" if DRY_RUN else "evaluate · three contenders on test (once)"
WINNER_TAG = "dry_run_winner" if DRY_RUN else "stage4_winner"
answers = importlib.import_module("04_evaluate_answers")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)


def attempt(value, name):
    # Your answer; if it still has a blank, a reminder (or, when checking the notebook end to end, the answer file's).
    if "___" not in str(value):
        return value
    if FILL:
        return getattr(answers, name)
    print("Fill in the ___ first, then run this cell again.")
    return None


print(f"scoring split: {SPLIT}" + ("  (DRY RUN: dev stands in for test; nothing here is a result)" if DRY_RUN else "  (the once-only look)"))

## 0. Setup: the frozen data, the honest test read, the candidate from MLflow (*Dev, test and holdout sets*)

The test questions and the course's paragraphs are read back from **stage 1's newest run** by name, and each table's
**digest** (a short code worked out from every row, logged by MLflow with the dataset) is checked against the one stage 1
logged: if a single question moved, the cell stops.

**The course has changed since stage 1, and this notebook scores stage 1's snapshot on purpose.** Two content commits
merged after stage 1 froze the course reworded some Learn paragraphs and added a few. The models were trained and chosen on
stage 1's snapshot, so they are judged on it too: the same 744 paragraphs, the same questions. Two things are checked
instead of the old "fingerprint must equal" line: the **split** is unchanged (a fresh deal of today's course puts every
lesson where `data/splits.json` has it), and the count of test questions whose wording changed in today's course. Both
fingerprints (stage 1's and today's) are logged, so the record says which content was scored. Stage 6 re-embeds the live
course when the winner is wired into the coach.

In [ ]:
tracking.connect()
client = mlflow.MlflowClient()
experiment = client.get_experiment_by_name(tracking.EXPERIMENT)
s1 = t.load_stage1()   # pairs_train, pairs_dev, honest_dev, corpus: each checked against stage 1's digest
stage1, digests = s1["run"], s1["digests"]


def load(name, targets="lesson"):
    # The rows stage 1 registered, read back from its run and registered again here: the digest must be the same one.
    rows = json.loads(Path(mlflow.artifacts.download_artifacts(run_id=stage1.info.run_id, artifact_path=f"{name}.json")).read_text())
    ds = tracking.table_dataset(rows, name, targets=targets)
    assert ds.digest == digests[name], f"{name}: digest {ds.digest} is not stage 1's {digests[name]}: the data moved, stop and look"
    return rows, ds


pairs_eval, ds_pairs = load(f"pairs_{SPLIT}")
honest_eval, ds_honest = load(f"honest_{SPLIT}")
corpus, ds_corpus = s1["corpus"], s1["datasets"]["corpus"][0]
splits = r.load_splits()
frozen_at_stage1 = r.load_splits(mlflow.artifacts.download_artifacts(run_id=stage1.info.run_id, artifact_path="splits.json"))
assert splits == frozen_at_stage1, "data/splits.json is not the split stage 1 logged: stop and look"

data = build_data()                                   # today's course, only to check the split and count changed wording
fresh_deal = r.split_lessons(data)
moved = sorted(lid for lid in set(fresh_deal) | set(splits) if fresh_deal.get(lid) != splits.get(lid))
assert not moved, f"today's course deals {len(moved)} lessons differently ({moved[:5]}): the split moved, rerun stage 1 first"
pairs_now = r.build_pairs(data)
live = {p["id"]: p["question"] for p in pairs_now}
changed_wording = sum(live.get(p["id"]) != p["question"] for p in pairs_eval)
fingerprint_now = r.content_fingerprint(r.build_corpus(data), pairs_now)
fingerprint_stage1 = stage1.data.params["content_fingerprint"]
lessons_eval = {p["lesson"] for p in pairs_eval}
print(f"stage 1 run {tracking.short(stage1.info.run_id)}: pairs_{SPLIT} {len(pairs_eval)} ({ds_pairs.digest}), honest_{SPLIT} {len(honest_eval)} "
      f"({ds_honest.digest}), corpus {len(corpus)} paragraphs ({ds_corpus.digest})")
print(f"{SPLIT}: {len(lessons_eval)} lessons in {len({p['module'] for p in pairs_eval})} modules; by kind {dict(Counter(p['chip'] or p['source'] for p in pairs_eval))}")
print(f"split: unchanged in today's course (0 of {len(splits)} lessons move; {r.split_counts(splits)}). Course fingerprint: stage 1 "
      f"{fingerprint_stage1}, today {fingerprint_now} ({'the same' if fingerprint_now == fingerprint_stage1 else 'changed since: stage 1 snapshot scored'}); "
      f"{changed_wording} of the {len(pairs_eval)} {SPLIT} questions are worded differently today")

**Has every learner-worded test question been read?** Each rewrite in `honest_test` must have been read against its
original question, for the same meaning and the same lesson, *before* any test score exists: reading them afterwards, with
the scores in view, would let the scores decide which rewrites to keep. The check reads two records: stage 1's `checked`
flag on each row (the one the digest covers) and the rewrite cache (`data/paraphrases.json`), which says who read it.

If any were unread, the agent working through this notebook (Claude, at the owner's request) would read each one now
against its original, write the verdict into `VERDICTS` below (`True`: same meaning, same lesson; `False`: drop it), and
`retrieval.record_checks(..., by="Claude")` would record them before the cell goes on. A rewrite marked `False` is dropped.

In [ ]:
store = json.loads((tracking.DATA / "paraphrases.json").read_text())


def cache_entry(h):
    # The rewrite's entry in the cache: which model made it and who read it.
    orig, i = h["id"].rsplit("~", 1)
    return [e for e in store.get(orig, []) if e.get("model") == r.PARAPHRASE_MODEL][int(i)]


VERDICTS = {}   # honest-test id -> True (same meaning, same lesson) or False (drop); filled only for rewrites not yet read

unread = [h["id"] for h in honest_eval if not h["checked"] or not cache_entry(h).get("checked")]
rejected = [h["id"] for h in honest_eval if cache_entry(h).get("checked") and not cache_entry(h).get("ok")]
if unread and not DRY_RUN:
    missing = [hid for hid in unread if VERDICTS.get(hid) is None]
    assert not missing, f"{len(missing)} {SPLIT} rewrites are unread ({missing[:5]}): read each against its original and fill VERDICTS first"
    state = r.record_checks(VERDICTS, by="Claude")
    store = json.loads((tracking.DATA / "paraphrases.json").read_text())
    dropped = {hid for hid, ok in VERDICTS.items() if ok is False}
    honest_eval = [dict(h, checked=1) for h in honest_eval if h["id"] not in dropped]
    ds_honest = tracking.table_dataset(honest_eval, f"honest_{SPLIT} · read", targets="lesson")
    print(f"read now by Claude: {len(VERDICTS)}; dropped {len(dropped)}; new dataset {ds_honest.digest}")
assert DRY_RUN or not rejected, f"rewrites marked as changing the meaning are still in the set: {rejected[:5]}"
read_by = Counter(cache_entry(h).get("checked_by", "a person") for h in honest_eval if cache_entry(h).get("checked"))
honest_state = (f"{len(honest_eval)} {r.PARAPHRASE_MODEL} rewrites of the {SPLIT} Check and practice questions; read: "
                + ", ".join(f"{n} by {who}" for who, n in read_by.items()) + f"; unread {len(honest_eval) - sum(read_by.values())}")
originals_ids = {h["orig"] for h in honest_eval}
originals_eval = [p for p in pairs_eval if p["id"] in originals_ids and p["source"] in ("check", "more")]
assert len(originals_eval) == len(honest_eval), "each learner-worded question must have exactly one course-worded original"
course_worded_all = [p for p in pairs_eval if p["source"] in ("check", "more")]
eval_sets = {"honest": honest_eval, "originals": originals_eval, "all": pairs_eval}
print(honest_state + (" (dry run: dev's few unread rewrites are kept, as stage 3 kept them)" if DRY_RUN and unread else ""))
print(f"the three {SPLIT} question sets: honest {len(honest_eval)} (learner-worded), originals {len(originals_eval)} (the same questions as the "
      f"course wrote them), all {len(pairs_eval)} (with the prepared chips, reported by group, outside the bar). "
      f"{len(course_worded_all) - len(originals_eval)} course-worded questions have no rewrite (theirs changed the meaning and was dropped at stage 1).")

**The candidate, from MLflow, not from a folder.** Stage 3 tagged exactly one run `candidate=stage4` and logged its best
checkpoint as an MLflow **logged model** (the model saved with how to load it and what goes in and out; glossary: logged
model). It is loaded from there, by its model id, so the thing tested here is the thing stage 5 registers. A copy loaded from
a folder on disk could have been overwritten by a later run with the same name.

Then a check that it really is the model that was chosen: scored on the **dev** questions with the same code, it must give
the dev score its run logged, to within one question, because a different batch order can change the last digit of two
near-equal similarities and swap a tie, and one question is the smallest step a score can move. Dev is not test; this reads nothing new.

In [ ]:
cands = client.search_runs([experiment.experiment_id], filter_string="tags.candidate = 'stage4' and attributes.status = 'FINISHED'")
assert len(cands) == 1, f"{len(cands)} runs carry candidate=stage4: exactly one may"
cand_run = cands[0]
logged = [m for m in client.search_logged_models([experiment.experiment_id], filter_string=f"source_run_id = '{cand_run.info.run_id}'")
          if m.status == LoggedModelStatus.READY]
assert len(logged) == 1, f"the candidate run has {len(logged)} ready logged models: exactly one may"
cand_model = logged[0]
CANDIDATE = r.logged_model_setup(f"bge-large · fine-tuned · {cand_run.info.run_name.removeprefix('train · bge-large · ')}", cand_model.model_uri)
cand_st = r.local_model(CANDIDATE.model)
assert cand_st.max_seq_length == 512, "the logged model must cut inputs at the makers' 512, as the champion does"
CONTENDERS = (r.CHAMPION, CANDIDATE, r.GEMMA)
CHALLENGERS = (CANDIDATE, r.GEMMA)
print(f"candidate: run \"{cand_run.info.run_name}\" ({tracking.short(cand_run.info.run_id)}), best step {cand_run.data.tags.get('best_step')}, "
      f"logged model {cand_model.model_id[:10]} ({cand_model.status.name}), weights {r.size_mb(CANDIDATE)} MB, max length {cand_st.max_seq_length}")

stage2 = tracking.find_run(tracking.EXPERIMENT, "baselines · champion and the bar (dev)")
stage3 = tracking.find_run(tracking.EXPERIMENT, "train · bge-large on the course (dev)")
assert stage2.data.params["stage1_run_id"] == stage1.info.run_id, "stage 2's bar was set on an older split"
assert stage3.data.params["stage1_run_id"] == stage1.info.run_id, "stage 3 trained on an older split"
assert stage3.data.tags.get("candidate_run") == tracking.short(cand_run.info.run_id), "the candidate is not the one stage 3's newest pass picked"
kids2 = {k.info.run_name: k for k in client.search_runs([experiment.experiment_id], filter_string=f"tags.mlflow.parentRunId = '{stage2.info.run_id}'")}
champ_dev_run, gemma_dev_run = kids2[f"{r.CHAMPION.name} · dev"], kids2[f"{r.GEMMA.name} · dev"]


# The dev numbers each contender carried into this notebook, read from the runs that logged them (stage 2 and stage 3).
dev = {r.CHAMPION.name: {"honest": champ_dev_run.data.metrics["honest.recall_at_5"], "originals": champ_dev_run.data.metrics["originals.recall_at_5"],
                         "honest_hit_at_1": champ_dev_run.data.metrics["honest.hit_at_1"], "n": int(champ_dev_run.data.metrics["honest.n"])},
       r.GEMMA.name: {"honest": gemma_dev_run.data.metrics["honest.recall_at_5"], "originals": gemma_dev_run.data.metrics["originals.recall_at_5"],
                      "honest_hit_at_1": gemma_dev_run.data.metrics["honest.hit_at_1"], "n": int(gemma_dev_run.data.metrics["honest.n"])},
       CANDIDATE.name: {"honest": cand_run.data.metrics["best.honest_recall_at_5"], "originals": cand_run.data.metrics["best.originals_recall_at_5"],
                        "honest_hit_at_1": cand_run.data.metrics["best.honest_hit_at_1"], "n": len(s1["honest_dev"])}}
pd.DataFrame(dev).T.round(3)

In [ ]:
t0 = time.monotonic()
same_model = r.evaluate_setup(CANDIDATE, corpus, {"honest": s1["honest_dev"], "originals": s1["originals_dev"]}, splits, progress=False)
identity = {name: same_model["sets"][name]["summary"]["recall_at_5"] for name in ("honest", "originals")}
gap = max(abs(identity[name] - dev[CANDIDATE.name][name]) for name in identity)
print(f"the logged model on dev: honest {identity['honest']:.3f}, originals {identity['originals']:.3f}; its run logged "
      f"{dev[CANDIDATE.name]['honest']:.3f} and {dev[CANDIDATE.name]['originals']:.3f} at step {cand_run.data.tags.get('best_step')} "
      f"(largest difference {gap:.3f}, {round(gap * len(s1['honest_dev']))} questions) · {time.monotonic() - t0:.0f} s")
assert gap <= 1.5 / len(s1["honest_dev"]), "the logged model does not give its run's dev score: it may not be the model that was chosen; stop and look"

**One parent run, and the same-look check.** If this notebook has run on test before, its first pass is found and compared.
A pass counts as a look as soon as it has logged `test_opened_at` (section 3 logs it just before the first test question is
embedded), **whatever its status**: a pass that opened test and then crashed (FAILED), was stopped (still RUNNING or KILLED)
or was deleted still saw test scores. The earliest such run is the first look. The same candidate run, the same logged model
and the same bar mean this pass is the same look again; anything different is refused, because it would be a new look at a
set that has already been seen.

In [ ]:
# Every earlier pass that opened test, whatever its status (FAILED, RUNNING, KILLED, deleted): a crash after opening is still a look.
looks_before = sorted((x for x in client.search_runs([experiment.experiment_id], filter_string="tags.`lab.stage` = '04_evaluate'",
                                                     run_view_type=ViewType.ALL, max_results=5000)
                       if x.data.params.get("test_opened_at")), key=lambda x: x.info.start_time)
first_look = looks_before[0] if looks_before and not DRY_RUN else None
if first_look is not None:
    state = first_look.info.status + ("" if first_look.info.lifecycle_stage == "active" else ", deleted")
    if state != "FINISHED":
        print(f"note: the first look, run {tracking.short(first_look.info.run_id)}, is {state}: it opened test and did not finish "
              "cleanly, but it saw test scores, so it is still the first look and this pass is held to it")
    assert first_look.data.params.get("candidate_run_id") == cand_run.info.run_id, ("the candidate changed after test was opened: "
                                                                                    "that is a new look, which the once-only rule forbids")
    assert first_look.data.params.get("candidate_model_id") == cand_model.model_id, "the candidate's logged model changed after test was opened"
    print(f"test was first opened by run {tracking.short(first_look.info.run_id)} ({state}) on {first_look.data.params.get('test_opened_at')}; "
          f"{len(looks_before)} earlier pass{'es' if len(looks_before) != 1 else ''} opened it. This pass re-scores the same frozen models on the "
          "same questions (the same look again, not a new one); the bar is checked below")
else:
    print("no earlier look at test: this is the first" if not DRY_RUN else
          f"dry run: the same-look check applies to the real pass only ({len(looks_before)} real looks at test so far)")

parent = mlflow.start_run(run_name=RUN_NAME, description="(the full description is written at the end of the notebook, once the numbers are known)")
tracking.guard_run()   # a cell that raises ends the run as FAILED, not as an empty FINISHED shell
mlflow.set_tags({"kind": "coach_retrieval_lab", "lab.stage": "04_evaluate", "git_commit": tracking.git_commit(), "split_scored": SPLIT,
                 "purpose": "dry run on dev: checks the code, not a result" if DRY_RUN else "the once-only look at test",
                 "champion": r.CHAMPION.name, "challengers": " | ".join(c.name for c in CHALLENGERS),
                 "candidate_run": tracking.short(cand_run.info.run_id), "candidate_model": cand_model.model_id[:10],
                 "honest_read_by": "Claude, a model, at the owner's request (not a person)",
                 "stage1_run": tracking.short(stage1.info.run_id), "stage2_run": tracking.short(stage2.info.run_id),
                 "stage3_run": tracking.short(stage3.info.run_id),
                 **{f"computer.{k}": str(v) for k, v in tracking.this_computer().items()}})
mlflow.log_params({"stage1_run_id": stage1.info.run_id, "stage2_run_id": stage2.info.run_id, "stage3_run_id": stage3.info.run_id,
                   "candidate_run_id": cand_run.info.run_id, "candidate_model_id": cand_model.model_id, "candidate_name": CANDIDATE.name,
                   "content_fingerprint.stage1": fingerprint_stage1, "content_fingerprint.now": fingerprint_now,
                   "questions_worded_differently_now": changed_wording, "split": SPLIT, "k": r.K, "seed": r.SEED,
                   "n.honest": len(honest_eval), "n.originals": len(originals_eval), "n.all": len(pairs_eval), "n.lessons": len(lessons_eval)})
for ds, context in ((ds_pairs, SPLIT), (ds_honest, SPLIT), (ds_corpus, "corpus")):
    mlflow.log_input(ds, context=context)
mlflow.log_metrics({f"identity_check.dev.{name}.recall_at_5": v for name, v in identity.items()})
you = tracking.Guesses()
T0 = time.monotonic()
print(f"MLflow parent run {tracking.short(parent.info.run_id)} in experiment {parent.info.experiment_id}; answers filled in: {FILL}")

## 1. The bar, written before the test is opened (*Testing many variants*)

> **In plain words: pre-registration.** Writing down the measure and the decision rule *before* the results come in, so the
> result cannot shape the rule, is **pre-registration** (glossary: pre-registration). The danger it guards against is quiet:
> once you have seen that a challenger won by 0.03 with p 0.04, it is very easy to decide that 0.05 was the right line all
> along. So the bar below is logged to MLflow as the parent run's tag `the_bar` and as params, with the time it was written,
> and only then is the first test question turned into a vector. Params and tags carry no time of their own, so two
> **metrics** mark the order: `bar_logged`, sent right after the bar, and `test_opened`, sent just before the first test
> question is embedded. Each metric value carries a timestamp in milliseconds, stamped by this computer as it is sent, so
> the run's metric history alone shows which came first. Section 3 checks the tag is there, and that the two timestamps are
> in that order, before it starts.

The bar is stage 2's bar, with one change: there are now **two challengers**, the fine-tune and embeddinggemma as shipped,
so the 5% chance of a fluke is shared between them.

> **In plain words: why each challenger gets 0.025.** Each test at the 0.05 line has about a 1-in-20 chance of a fluke when
> nothing really differs. Two challengers tested against the champion on the same questions are two such chances, so the
> chance that *at least one* wins by luck is nearly double, about 1 in 10 (the **multiple comparisons problem**, glossary:
> multiple comparisons problem, and *Testing many variants*). Sharing the 0.05 out, 0.05 / 2 = 0.025 for each, is the
> **Bonferroni correction** (glossary: Bonferroni correction): it keeps the chance of at least one fluke across both at about
> 5%. Only the McNemar line is shared out; the bootstrap range stays the 95% one stage 2 wrote, and a challenger needs both,
> so the stricter of the two decides.

Each part of the bar has its one-sentence reason beside it.

In [ ]:
P_LINE = 0.05                          # the field's usual line (stage 2: "why 0.05")
P_EACH = P_LINE / len(CHALLENGERS)     # Bonferroni: the 0.05 shared by the challengers tested on the same questions
SPEED_REFERENCE = {CANDIDATE.name: r.CHAMPION, r.GEMMA.name: r.CHAMPION_OLLAMA}   # each challenger is timed against the champion served the same way
n_h, n_o = len(honest_eval), len(originals_eval)
BAR = {
    "win": (f"On the {n_h} learner-worded {SPLIT} questions (honest_{SPLIT}), the challenger's recall@5 beats the champion's ({r.CHAMPION.name}) with the "
            f"paired bootstrap 95% range of (challenger minus champion) above zero AND McNemar's p under {P_EACH:g}."),
    "win.why": ("On Databricks the champion is a pay-per-token endpoint with no upkeep and any challenger is a served model billed by the hour, so only "
                f"a win on learner wording that cannot be luck pays for it, and two challengers share the 0.05 because two tries on the same questions "
                f"nearly double the chance of one fluke."),
    "forgetting": (f"On the {n_o} course-worded {SPLIT} questions (the same Check and practice questions as the course wrote them), the paired bootstrap 95% "
                   f"range of (challenger minus champion) recall@5 includes zero or sits above it, AND McNemar does not show a champion win at {P_LINE:g}."),
    "forgetting.why": ("A gain on new wording bought by losing questions the champion already finds is catastrophic forgetting; the check is paired because "
                       "separate ranges can hide a one-sided loss, and it keeps the plain 0.05 because sharing out guards against false wins, and a "
                       "stricter line here would only make a real loss easier to miss."),
    "speed": (f"The challenger's median milliseconds per question over 5 timed passes on the same {n_o} course-worded questions, batch 32, after one "
              f"untimed warm-up pass for every setup, with the order of the setups turned by one place each pass, in this session, is no higher than "
              f"the champion's median plus the champion's middle spread (4th-fastest pass minus 2nd-fastest), the champion served the same way as the "
              f"challenger: sentence-transformers on this computer for the fine-tune, Ollama for embeddinggemma. A re-run reads the first look's "
              f"passes back instead of timing again."),
    "speed.why": ("A difference smaller than the pass-to-pass noise cannot be told from the champion, and stage 2 saw the serving path alone move the "
                  "champion from about 5 to 12 ms a question, so a model is timed against the champion served its way; the warm-up and the turning "
                  "order take the cost of going first off every setup, and the middle spread is used because one cold pass would otherwise widen "
                  "the champion's window and let a slower challenger through."),
    "both_clear": ("If both challengers clear the bar, the one whose gap range on the learner-worded questions has the larger lower end wins; if the two "
                   "lower ends are within one question of each other, the smaller model file wins."),
    "both_clear.why": ("The lower end is the gain the range still vouches for, so it is the cautious pick; one question is the smallest difference "
                       "the test questions can show, so a gap of one is treated as a tie; and both would be served by the hour on Databricks, where a "
                       "smaller model needs less memory and so a cheaper endpoint."),
    "neither": "If neither clears the bar, the champion stays: bge-large as shipped, with its search prompt.",
    "neither.why": "The champion is already served pay-per-token with no upkeep, so a challenger that has not shown a win gives no reason to change it.",
}
the_bar = " ".join(f"[{part}] {text}" for part, text in BAR.items())
bar_digest = hashlib.sha256(the_bar.encode()).hexdigest()[:12]
assert "results" not in globals(), f"{SPLIT} scores already exist in this session: restart the kernel and run from the top, so the bar comes first"
written_at = datetime.now(timezone.utc).isoformat(timespec="milliseconds")
if first_look is not None:
    assert first_look.data.params.get("bar.digest") == bar_digest, "the bar differs from the one written before test was first opened: refused"
mlflow.set_tag("the_bar", the_bar)
mlflow.log_metric("bar_logged", 1)   # a metric carries its own timestamp, in milliseconds: the record of when the bar was in place
mlflow.log_params({**{f"bar.{part}": text for part, text in BAR.items()}, "bar.p_each": P_EACH, "bar.p_line": P_LINE,
                   "bar.challengers": " | ".join(c.name for c in CHALLENGERS), "bar.champion": r.CHAMPION.name,
                   "bar.speed_reference": " | ".join(f"{k} -> {v.name}" for k, v in SPEED_REFERENCE.items()),
                   "bar.written_at": written_at, "bar.digest": bar_digest})
for part, text in BAR.items():
    print(f"{part:15} {text}")
print(f"\nTHE BAR was written at {written_at} (UTC), logged as the tag 'the_bar' and the params bar.* of run {tracking.short(parent.info.run_id)} "
      f"(digest {bar_digest}). No {SPLIT} question has been embedded yet.")

### Exercise 1: which p line applies?

Two challengers are tested against the champion on the same test questions, and the usual line is 0.05. Under what p must
*each* challenger's McNemar test come for its win to count?

In [ ]:
P_FOR_EACH = "___"   # a number between 0 and 1

In [ ]:
mine = attempt(P_FOR_EACH, "P_FOR_EACH")
if mine is not None:
    assert abs(float(mine) - 0.025) < 1e-9, ("Not quite: the 0.05 is shared out between the two challengers, so each gets half of it. "
                                              "0.05 for each would nearly double the chance that one of them wins by luck.")
    print("Right: 0.025. Each challenger gets half of the 0.05; Exercise 3 works out what that buys.")

### Exercise 2: why before?

Why is the bar logged to MLflow *before* the first test question is embedded? Pick one:

* `"a"`: so the bar cannot be quietly changed to fit the scores once they are known
* `"b"`: because MLflow needs params to be logged before metrics
* `"c"`: so the test runs faster

In [ ]:
WHY_BEFORE = "___"   # "a", "b" or "c"

In [ ]:
mine = attempt(WHY_BEFORE, "WHY_BEFORE")
if mine is not None:
    assert str(mine).strip().lower() == "a", ("Not quite: MLflow takes params and metrics in any order, and speed has nothing to do with it. "
                                              "The point is that a rule written after the result can be bent to fit it.")
    print(f"Right: a. The run records the bar at {written_at}; section 3 sends a metric the moment the test is opened, and the two "
          "metrics' own timestamps (in milliseconds) show the order, so anyone can check it from the run's metric history.")

### Exercise 3: the chance of at least one fluke

Two win tests, each at the shared-out line of 0.025. Using the rule in `r.family_wise_error_rate` (one minus 0.975 multiplied
by itself once per test), what is the chance that at least one of them comes out under the line by luck alone, when nothing
really differs? Three decimal places.

In [ ]:
FLUKE_CHANCE = "___"   # a number between 0 and 1, e.g. 0.123

In [ ]:
mine = attempt(FLUKE_CHANCE, "FLUKE_CHANCE")
if mine is not None:
    assert abs(float(mine) - r.family_wise_error_rate(2, 0.025)) < 0.002, ("Not quite: the chance that one test does not fluke is 0.975; that both do not "
                                                                            "is 0.975 × 0.975; one minus that is the chance that at least one does.")
    print(f"Right: {r.family_wise_error_rate(2, 0.025):.3f}, just under the 0.05 a single test would have. That is what the sharing out buys.")

## 2. Your guesses, before any test number

The cell below prints the score each contender carried in from the learner-worded **dev** questions, read from the runs
that logged them (stage 2 for the champion and embeddinggemma, stage 3 for the candidate). Write your guesses down before
section 3 runs.

* What will the champion's recall@5 be on the learner-worded **test** questions?
* Which challenger clears the bar: `"candidate"`, `"embeddinggemma"`, `"both"` or `"neither"`?

In [ ]:
for name, d in dev.items():
    print(f"{name:45} dev honest recall@5 {d['honest']:.3f} (n {d['n']})")
you.guess("champion_recall_test_honest", None)   # e.g. 0.75
WINNER_GUESS = None                              # e.g. "candidate", "embeddinggemma", "both" or "neither"
you.guess("which_challenger_clears", WINNER_GUESS)

### Exercise 4: what the winner's curse predicts

The candidate was the best of four training runs on the dev questions, at the best of several checkpoints, and
embeddinggemma was noticed because it came out ahead on dev. Before section 3 opens the test: on test, will a
challenger's gap over the champion most likely be `"larger"`, `"about the same"` or `"smaller"` than its gap on dev?

In [ ]:
WINNERS_CURSE = "___"   # "larger", "about the same" or "smaller"

In [ ]:
mine = attempt(WINNERS_CURSE, "WINNERS_CURSE")
if mine is not None:
    assert str(mine).strip().lower() == "smaller", ("Not quite: part of why something came out best on dev is luck on those dev questions, and "
                                                    "that luck does not come with it to new questions.")
    print("Right: smaller. The best of several tries looks better than it is on the set it was picked on; that is the winner's curse.")

## 3. Open the test: three contenders, once (*Measuring retrieval*, *Margins near 0% and 100%*)

> **In plain words: what is counted.** Every contender searches the **whole course** (all 744 Learn paragraphs) for every
> question. Three measures, as in stages 2 and 3:
>
> * **recall@5**: the share of questions where *any* paragraph of the right lesson is among the five closest. With one right
>   lesson per question that is the **hit rate at 5** (*Measuring retrieval*; glossary: Recall@k). The bar's number.
> * **hit@1**: the stricter share where the *closest* paragraph is the right lesson's (glossary: hit@1).
> * **MRR** (mean reciprocal rank): 1 divided by the place of the first right paragraph, averaged; 1.0 means always first
>   (glossary: MRR).
> * **The Wilson range**: where the true share plausibly sits given how many questions there are, caught about 95 times in
>   100 (*Margins near 0% and 100%*; glossary: Wilson interval, confidence interval). 81 questions give a range about 18
>   points wide, so a single share here says much less than it looks.

Three sets of test questions, all logged: **honest** (the learner-worded rewrites, the bar's set), **originals** (the same
questions as the course wrote them, the forgetting check) and **all** (with the prepared chips, reported by group only:
option chips name no topic and stay out of the bar, as stage 2 decided). One **child run** per contender holds its numbers,
its ranked rows and the test datasets as inputs.

In [ ]:
assert client.get_run(parent.info.run_id).data.tags.get("the_bar"), "the bar is not logged yet: run section 1 first"
opened_at = datetime.now(timezone.utc).isoformat(timespec="milliseconds")
assert opened_at >= written_at
mlflow.log_param("test_opened_at" if not DRY_RUN else "dry_run_opened_at", opened_at)
OPENED_METRIC = "test_opened" if not DRY_RUN else "dry_run_opened"
mlflow.log_metric(OPENED_METRIC, 1)   # its timestamp, in milliseconds, is the record of when the first question was about to be embedded
marks = {m: client.get_metric_history(parent.info.run_id, m)[0].timestamp for m in ("bar_logged", OPENED_METRIC)}
assert marks["bar_logged"] < marks[OPENED_METRIC], "the record does not show the bar before the opening: stop and look"
print(f"bar written {written_at}; {SPLIT} opened {opened_at} (UTC); in the metric history, bar_logged comes "
      f"{marks[OPENED_METRIC] - marks['bar_logged']} ms before {OPENED_METRIC}")
t0 = time.monotonic()
results = {s.name: r.evaluate_setup(s, corpus, eval_sets, splits) for s in CONTENDERS}
seconds_scoring = time.monotonic() - t0
mlflow.log_metric("seconds_scoring", round(seconds_scoring, 1))
print(f"{len(results)} contenders × {sum(len(q) for q in eval_sets.values())} questions in {seconds_scoring:.0f} s")

In [ ]:
honest = r.results_table(results, "honest")
originals = r.results_table(results, "originals")
you.reveal("champion_recall_test_honest", honest.loc[r.CHAMPION.name, "recall@5"])
mlflow.log_table(honest.reset_index(), f"tables/{SPLIT}_honest.json")
mlflow.log_table(originals.reset_index(), f"tables/{SPLIT}_originals.json")
print(f"learner-worded {SPLIT} questions (honest, n {n_h}):")
display(honest.round(3))
print(f"course-worded {SPLIT} questions (originals, the same {n_o} questions as the course wrote them):")
display(originals.round(3))

**By group, on all the test questions** (*Report each group, weighted by real traffic*). Each cell is the share with its
Wilson range; option chips are shown and stay out of the bar.

In [ ]:
groups_all = r.by_group_table(results, "all")
mlflow.log_table(groups_all.reset_index().rename(columns={"index": "setup"}), f"tables/{SPLIT}_all_by_group.json")
groups_all

**The weakest module** (*Embedding models have blind spots*). A module with one or two test questions scores 0 or 1 by
chance, so the weakest is read only among modules with at least five learner-worded questions, with the count beside it, and
the full table is logged.

In [ ]:
mods = {}
for name, res in results.items():
    for m in res["sets"]["honest"]["summary"]["by_module"]:
        mods.setdefault(m["module"], {"n": m["n"]})[name] = m["found"]
by_module = pd.DataFrame(mods).T.sort_values([r.CHAMPION.name, "n"]).astype(int)
mlflow.log_table(by_module.reset_index().rename(columns={"index": "module"}), f"tables/{SPLIT}_honest_by_module.json")
AT_LEAST = 5
enough = by_module[by_module["n"] >= AT_LEAST]
small = by_module[by_module["n"] < AT_LEAST]["n"]
print(f"{len(by_module)} modules in honest_{SPLIT}; {len(enough)} have {AT_LEAST} or more questions ({int(enough['n'].sum())} of {n_h} questions)"
      + (f"; the other {len(small)} have {int(small.min())} to {int(small.max())} each" if len(small) else ""))
weakest = {}
for name in results:
    ws = t.weakest(results[name]["sets"]["honest"]["summary"], at_least=AT_LEAST)
    weakest[name] = ws[0] if ws else "(no module with 5+ questions)"
    print(f"  {name:45} weakest with {AT_LEAST}+ questions: {', '.join(ws) or '-'}")
mlflow.set_tags({f"weakest_5plus.honest.{i}": f"{name}: {w}" for i, (name, w) in enumerate(weakest.items())})
by_module

**Speed and size** (*Latency that users feel*). Every setup first gets one untimed **warm-up pass** (the benchmarking
sense of warm-up: an untimed run so the model is loaded and its caches are full before the clock starts; the glossary's
*warm-up* is the training sense, a different thing). Then five timed rounds over the same course-worded test questions,
batch 32, all in this session, with the order of the four setups turned by one place each round, so none of them always
goes first (`retrieval.speed_rounds`; the order is logged). The **median** of the five passes (the middle one once they are
in order; glossary: median) is what is compared. The noise on this computer is the **middle spread**, the 4th-fastest pass
minus the 2nd-fastest: with five passes this is the **interquartile range** (the spread of the middle half of the
passes), which one slow pass cannot blow up. The full spread (slowest minus fastest) is shown beside it. The champion is
timed twice: on this computer (the fine-tune's path) and through Ollama (embeddinggemma's path), because the bar times each
challenger against the champion served the same way. On a re-run, the first look's own passes are read back instead of
timed again, so re-running cannot change the speed verdict. The dollar line is 0 here by construction; stage 8 adds
Databricks' prices.

In [ ]:
timing_questions = [p["question"] for p in originals_eval]
SPEED_SETUPS = [r.CHAMPION, CANDIDATE, r.CHAMPION_OLLAMA, r.GEMMA]
timed_by = next((x for x in looks_before if any(a.path == "speed_passes.json" for a in client.list_artifacts(x.info.run_id))),
                None) if first_look is not None else None
if timed_by is not None:
    speed = json.loads(Path(mlflow.artifacts.download_artifacts(run_id=timed_by.info.run_id, artifact_path="speed_passes.json")).read_text())
    assert set(speed) == {s.name for s in SPEED_SETUPS}, "the first look timed different setups: stop and look"
    print(f"timing read back from run {tracking.short(timed_by.info.run_id)}, the first look that timed: not measured again, "
          "so re-running cannot change a speed verdict")
else:
    print("Ollama holds right now:", t.ollama_loaded_gb() or "nothing", "(another job's model in the same memory can slow every pass)")
    speed = r.speed_rounds(SPEED_SETUPS, timing_questions, passes=5, warmup=1)
    for i, order in enumerate(next(iter(speed.values()))["order_each_round"]):
        print(f"  round {i}: " + " -> ".join(order))
mlflow.set_tag("speed_measured", f"read back from run {tracking.short(timed_by.info.run_id)}" if timed_by is not None else "this pass")
sizes = {s.name: r.size_mb(s) for s in SPEED_SETUPS}
mlflow.log_dict(speed, "speed_passes.json")
speed_table = pd.DataFrame([{"setup": name, "path": "Ollama" if "ollama" in name else "this computer (sentence-transformers)",
                             "ms/question median": sp["ms_per_question_median"], "middle spread": sp["ms_middle_spread"],
                             "fastest": sp["ms_per_question_min"], "slowest": sp["ms_per_question_max"], "full spread": sp["ms_spread"],
                             "model MB": sizes[name], "usd here": 0.0,
                             "on Databricks": "pay-per-token" if name in (r.CHAMPION.name, r.CHAMPION_OLLAMA.name) else "served, by the hour"}
                            for name, sp in speed.items()]).set_index("setup")
mlflow.log_table(speed_table.reset_index(), "tables/speed_and_size.json")
speed_table

In [ ]:
child_runs = {}
CAVEAT = (f"Caveats: the learner-worded set is {honest_state}, model-made and read by a model; option chips are reported by group and left out of the bar; "
          f"scored on stage 1's frozen snapshot of the course (fingerprint {fingerprint_stage1}; today's is {fingerprint_now}).")
for setup in CONTENDERS:
    res = results[setup.name]
    h, o, a = res["sets"]["honest"], res["sets"]["originals"], res["sets"]["all"]
    role = "champion" if setup is r.CHAMPION else "challenger"
    where = (f"loaded from its logged MLflow model {cand_model.model_id[:10]} (run {tracking.short(cand_run.info.run_id)}, best step "
             f"{cand_run.data.tags.get('best_step')}), never from a folder" if setup is CANDIDATE else
             "as its makers ship it, with its search prompt" + (" (the champion)" if setup is r.CHAMPION else ", served by Ollama"))
    description = (f"**{setup.name}** on the {SPLIT} split, {role}, one of three contenders under the parent run '{RUN_NAME}' (04_evaluate.ipynb). "
                   f"{'DRY RUN on dev: checks the code, not a result. ' if DRY_RUN else 'The once-only look at test, scored after the bar was logged at ' + written_at + '. '}"
                   f"Model: {where}. Main metrics: honest_{SPLIT}, {h['summary']['n']} learner-worded questions (the bar's set). `originals.*`: the same "
                   f"{o['summary']['n']} questions as the course wrote them (the forgetting check). `all.*`: all {a['summary']['n']} {SPLIT} pairs with the "
                   f"prepared chips. `speed.*`: median of 5 passes on the {n_o} originals, batch 32. Corpus: {len(corpus)} Learn paragraphs, embedded in "
                   f"{res['doc'].seconds} s; model load off the clock. " + CAVEAT)
    child = r.log_setup_run(setup, h["summary"], h["speed_cost"], [ds_honest, ds_pairs, ds_corpus], description, parent.info.run_id,
                            tags={"lab.stage": "04_evaluate", "question_set": f"honest_{SPLIT}", "role": role, "split_scored": SPLIT},
                            rows=h["rows"], run_name=f"{setup.name} · {SPLIT}")
    r.log_extra_set(child, "originals", o["summary"], o["speed_cost"], o["rows"])
    r.log_extra_set(child, "all", a["summary"], a["speed_cost"], a["rows"])
    with mlflow.start_run(run_id=child, nested=True):
        mlflow.log_metrics({f"speed.{k}": v for k, v in speed[setup.name].items() if isinstance(v, (int, float))} | {"model_size_mb": sizes[setup.name] or 0.0})
    child_runs[setup.name] = child
    print(f"{setup.name:45} child run {tracking.short(child)}")

## 4. Paired comparisons, then the bar applied in code (*Comparing on the same examples*, *Telling real gains from noise*)

> **In plain words.** The three contenders answered the *same* questions, so they are compared question by question
> (glossary: paired comparison). Questions both found, or both missed, say nothing about which is better; only the
> **disagreements** do (glossary: discordant pairs). **McNemar's test** asks how likely a split this lopsided among the
> disagreements would be if the two were really equal: that chance is the **p-value** (glossary: McNemar's test, p-value).
> The **paired bootstrap range** redraws the questions with replacement 2,000 times, keeping both models' answers to each,
> and keeps the middle 95% of the gaps (glossary: bootstrap). On the course-worded questions the same count is the
> regression check of *Reading regressions and improvements*: questions the champion found that the challenger now misses.

In [ ]:
champ_res = results[r.CHAMPION.name]
cmp = {(c.name, s): r.compare(champ_res["sets"][s]["rows"], results[c.name]["sets"][s]["rows"]) for c in CHALLENGERS for s in ("honest", "originals")}
cmp_table = pd.DataFrame([{"challenger": c, "set": s, "n": x["n"], "only champion found": x["only_first"], "only challenger found": x["only_second"],
                           "gap recall@5": x["gap_recall_at_5"], "gap low": x["gap_low"], "gap high": x["gap_high"], "McNemar p": x["p_value"]}
                          for (c, s), x in cmp.items()]).set_index(["challenger", "set"])
mlflow.log_table(cmp_table.reset_index(), "tables/paired_vs_champion.json")
mlflow.log_dict({f"{c} | {s}": x for (c, s), x in cmp.items()}, "paired_vs_champion.json")
looks = sum((x["only_first"] + x["only_second"]) > 0 for x in cmp.values())
fwer_wins = r.family_wise_error_rate(len(CHALLENGERS), P_EACH)
fwer_unshared = r.family_wise_error_rate(len(CHALLENGERS), P_LINE)
fwer_all = r.family_wise_error_rate(looks, P_LINE)
mlflow.log_metrics({"family_wise_error_rate.win_tests": fwer_wins, "family_wise_error_rate.win_tests_unshared": fwer_unshared,
                    "family_wise_error_rate.all_paired_tests_at_0.05": fwer_all, "paired_comparisons": looks})
display(cmp_table.round(3))
print(f"The two win tests at {P_EACH:g} each: the chance of at least one fluke (the family-wise error rate) is {fwer_wins:.3f}; "
      f"at 0.05 each it would have been {fwer_unshared:.3f}. All {looks} paired comparisons above, each read at 0.05, would give {fwer_all:.3f}: "
      f"that is why only the two win tests decide, at the shared-out line, and the forgetting checks only guard against harm.")

### Exercise 5: what does the forgetting check compare?

Which comparison does the forgetting check use? Pick one:

* `"a"`: the challenger's own Wilson range on the learner-worded questions
* `"b"`: the paired comparison of challenger against champion on the course-worded questions: the bootstrap range of
  (challenger minus champion) recall@5, and McNemar's test
* `"c"`: the challenger's training loss

In [ ]:
FORGETTING_NUMBER = "___"   # "a", "b" or "c"

In [ ]:
mine = attempt(FORGETTING_NUMBER, "FORGETTING_NUMBER")
if mine is not None:
    assert str(mine).strip().lower() == "b", ("Not quite: forgetting is losing what the champion could already do, which shows on the course's own "
                                              "wording, and only a paired count on the same questions can see a one-sided loss.")
    x = cmp[(CANDIDATE.name, "originals")]
    print(f"Right: b. For the fine-tune that is {x['gap_recall_at_5']:+.3f} [{x['gap_low']:+.3f}, {x['gap_high']:+.3f}] on {x['n']} course-worded "
          f"questions: {x['only_first']} found only by the champion, {x['only_second']} only by the fine-tune, McNemar p {x['p_value']:.4f}.")

**The margin.** 81 questions from 29 lessons is a small test. For each win test the code below also says how many
questions the verdict rests on: how many of the challenger's own finds would have to turn into misses to lose the McNemar
line, or how many more it would have needed to reach it.

In [ ]:
def mcnemar_margin(only_champion, only_challenger, line):
    # Questions the McNemar verdict rests on: fewer challenger-only finds that would lose the line, or more that would reach it.
    passes = only_challenger > only_champion and tracking.mcnemar(only_champion, only_challenger) < line
    k = 0
    if passes:
        while only_challenger - k > only_champion and tracking.mcnemar(only_champion, only_challenger - k) < line:
            k += 1
        return f"passes; {k} fewer question{'s' if k != 1 else ''} found only by the challenger would lose it"
    while not (only_challenger + k > only_champion and tracking.mcnemar(only_champion, only_challenger + k) < line):
        k += 1
    return f"fails; {k} more question{'s' if k != 1 else ''} found only by the challenger would have reached it"


verdicts = {}
for c in CHALLENGERS:
    w, f = cmp[(c.name, "honest")], cmp[(c.name, "originals")]
    sp, ref = speed[c.name], speed[SPEED_REFERENCE[c.name].name]
    limit = ref["ms_per_question_median"] + ref["ms_middle_spread"]
    parts = {
        "win: range above zero": (w["gap_low"] > 0, f"gap {w['gap_recall_at_5']:+.3f} [{w['gap_low']:+.3f}, {w['gap_high']:+.3f}] on {w['n']} learner-worded questions"),
        f"win: McNemar p under {P_EACH:g}": (w["only_second"] > w["only_first"] and w["p_value"] < P_EACH,
                                             f"{w['only_second']} only the challenger found against {w['only_first']}, p {w['p_value']:.4f} "
                                             f"({mcnemar_margin(w['only_first'], w['only_second'], P_EACH)})"),
        "forgetting: range includes zero or above": (f["gap_high"] >= 0, f"gap {f['gap_recall_at_5']:+.3f} [{f['gap_low']:+.3f}, {f['gap_high']:+.3f}] on {f['n']} course-worded questions"),
        f"forgetting: McNemar shows no champion win at {P_LINE:g}": (not (f["only_first"] > f["only_second"] and f["p_value"] < P_LINE),
                                                                     f"{f['only_first']} only the champion found against {f['only_second']}, p {f['p_value']:.4f}"),
        "speed: no slower than champion median + middle spread": (sp["ms_per_question_median"] <= limit,
                                                           f"{sp['ms_per_question_median']} ms against {ref['ms_per_question_median']} + {ref['ms_middle_spread']} = {limit:.2f} ms "
                                                           f"({SPEED_REFERENCE[c.name].name})"),
    }
    verdicts[c.name] = {"parts": parts, "clears": all(ok for ok, _ in parts.values())}
    print(f"\n{c.name}: {'CLEARS THE BAR' if verdicts[c.name]['clears'] else 'does not clear the bar'}")
    for part, (ok, why) in parts.items():
        print(f"  {'pass' if ok else 'FAIL'}  {part:52} {why}")
key = {CANDIDATE.name: "candidate", r.GEMMA.name: "embeddinggemma"}
mlflow.log_metrics({f"bar.{key[c]}.{i}.pass": float(ok) for c, v in verdicts.items() for i, (ok, _) in enumerate(v["parts"].values())}
                   | {f"bar.{key[c]}.clears": float(v["clears"]) for c, v in verdicts.items()})
mlflow.log_dict({c: {"clears": v["clears"], "parts": {p: {"pass": ok, "number": why} for p, (ok, why) in v["parts"].items()}} for c, v in verdicts.items()},
                "bar_verdicts.json")

## 5. The decision, and what would make it wrong (*Dev, test and holdout sets*)

The rule written in section 1, applied in code: one challenger clears, it wins; both clear, the larger lower end of the gap
range wins, and within one question of each other the smaller model file; neither clears, the champion stays.

**This is wrong if.** Say these before reading the decision below as a result:

* *the honest test is model-made and model-read.* `qwen3:14b` wrote the learner-worded questions and Claude, a model, read
  them for meaning. A rewrite a person would reject, or a learner would never type, counts the same as a good one. Only real
  typed questions (stage 7) settle how learners word things.
* *the candidate learned the rewriter's style.* The candidate's training rewrites and the honest test came from the same
  model and the same prompt (*Embedding models have blind spots*), so part of any gain on honest can be the fine-tune
  learning how `qwen3:14b` rewords, not how learners do. This is the one caveat that favours the candidate and not the
  others.
* *29 test lessons is small.* The verdict above rests on a handful of disagreements; the margin line in section 4 says how
  many questions would flip it. A "not shown" here is not "no difference" (*Enough to notice a difference*): a real gain of
  a few points could be too small for 81 questions to see.
* *dev was used for choosing.* The candidate's run, checkpoint and learning rate were all picked on dev, and gemma became a
  challenger because of dev; the dev-against-test table below shows how much of each gap survived.
* *the course has moved on.* Test was scored on stage 1's frozen snapshot; a few test paragraphs are worded differently in
  today's course. Stage 6 re-embeds the live course with the winner.

In [ ]:
clearing = [c for c in CHALLENGERS if verdicts[c.name]["clears"]]
if not clearing:
    winner, why = r.CHAMPION, "neither challenger cleared the bar, so the champion stays (the rule's last line)"
elif len(clearing) == 1:
    winner, why = clearing[0], "it is the only challenger that cleared every part of the bar"
else:
    lows = {c.name: cmp[(c.name, "honest")]["gap_low"] for c in clearing}
    a, b = sorted(clearing, key=lambda c: -lows[c.name])
    if lows[a.name] - lows[b.name] > 1 / n_h:
        winner, why = a, f"both cleared; its gap range has the larger lower end ({lows[a.name]:+.3f} against {lows[b.name]:+.3f})"
    else:
        winner = min(clearing, key=lambda c: sizes[c.name])
        why = f"both cleared with lower ends within one question ({lows[a.name]:+.3f}, {lows[b.name]:+.3f}); it is the smaller model file ({sizes[winner.name]} MB)"
decision = f"{'The champion stays' if winner is r.CHAMPION else 'The winner is ' + winner.name}: {why}."
decided_by = next((x for x in looks_before if x.data.tags.get("stage4_decision")), None) if first_look is not None else None
if decided_by is not None:
    # A re-run of the same look: the decision must come out as the first look's, and the winner tag stays where it was put.
    assert decision == decided_by.data.tags["stage4_decision"], ("this pass reaches a different decision from the first look's "
                                                                 f"(run {tracking.short(decided_by.info.run_id)}): refused, stop and look")
    mlflow.set_tags({"stage4_decision": decision, "same_look_as": tracking.short(decided_by.info.run_id)})
    winner_tagged_on = None
else:
    mlflow.set_tags({"stage4_decision": decision, "stage4_winner_name": winner.name} if not DRY_RUN else {"dry_run_decision": decision})
    with mlflow.start_run(run_id=child_runs[winner.name], nested=True):
        mlflow.set_tag(WINNER_TAG, "true")
    winner_tagged_on = child_runs[winner.name]
actual_clears = ("both" if len(clearing) == 2 else "neither" if not clearing else key[clearing[0].name])
mlflow.set_tag("actual.which_challenger_clears", actual_clears)
print(decision)
print(f"tagged {WINNER_TAG}=true on child run {tracking.short(winner_tagged_on)} ({winner.name} · {SPLIT})" if winner_tagged_on else
      f"the same decision as the first look (run {tracking.short(decided_by.info.run_id)}); no second {WINNER_TAG} tag is set, so stage 5 reads the first look's")
print(f"which challenger clears: you guessed {WINNER_GUESS!r}; it was {actual_clears!r}")

> **In plain words: the winner's curse.** When several tries are compared and the best is kept, part of why it came out
> best is luck on that set, so on new questions it usually does a little worse. The real term is **selection bias**, also
> called the **winner's curse** (glossary: selection bias (winner's curse); *Testing many variants*). The champion was *not*
> picked on dev: it was named before stage 2. So its own move from dev to test shows how much harder or easier the test
> lessons are, and the fair way to see the curse is each challenger's **gap over the champion**, dev against test: the split's
> difficulty cancels out of a gap.

In [ ]:
rows = []
for s in CONTENDERS:
    h, o = results[s.name]["sets"]["honest"]["summary"], results[s.name]["sets"]["originals"]["summary"]
    d = dev[s.name]
    rows.append({"contender": s.name, f"dev honest (n {d['n']})": d["honest"], f"{SPLIT} honest (n {h['n']})": h["recall_at_5"],
                 f"{SPLIT} range": f"[{h['recall_at_5_low']:.2f}, {h['recall_at_5_high']:.2f}]", "move": h["recall_at_5"] - d["honest"],
                 "dev gap over champion": d["honest"] - dev[r.CHAMPION.name]["honest"],
                 f"{SPLIT} gap over champion": h["recall_at_5"] - champ_res["sets"]["honest"]["summary"]["recall_at_5"],
                 "dev originals": d["originals"], f"{SPLIT} originals": o["recall_at_5"],
                 "dev hit@1 honest": d["honest_hit_at_1"], f"{SPLIT} hit@1 honest": h["hit_at_1"]})
dev_vs_test = pd.DataFrame(rows).set_index("contender")
dev_vs_test["gap shrank by"] = dev_vs_test["dev gap over champion"] - dev_vs_test[f"{SPLIT} gap over champion"]
mlflow.log_table(dev_vs_test.reset_index(), "tables/dev_vs_test.json")
mlflow.log_metrics({f"dev_vs_{SPLIT}.{key.get(s.name, 'champion')}.gap_shrank_by": float(dev_vs_test.loc[s.name, "gap shrank by"]) for s in CHALLENGERS}
                   | {f"dev_vs_{SPLIT}.champion.move": float(dev_vs_test.loc[r.CHAMPION.name, "move"])})
for s in CHALLENGERS:
    shrank = dev_vs_test.loc[s.name, "gap shrank by"]
    print(f"{s.name}: gap over the champion {dev_vs_test.loc[s.name, 'dev gap over champion']:+.3f} on dev, "
          f"{dev_vs_test.loc[s.name, f'{SPLIT} gap over champion']:+.3f} on {SPLIT}: "
          + (f"it shrank by {shrank:.3f} (about {round(shrank * n_h)} of {n_h} {SPLIT} questions), in the direction the winner's curse predicts "
                f"(a hunch, not a finding: one reading on {n_h} questions, with no range on the difference)"
             if shrank > 0 else f"it did not shrink here (a hunch too: one reading on {n_h} questions, with no range on the difference)"))
print(f"the champion moved {dev_vs_test.loc[r.CHAMPION.name, 'move']:+.3f} from dev to {SPLIT}: the {SPLIT} lessons look "
      f"{'harder' if dev_vs_test.loc[r.CHAMPION.name, 'move'] < 0 else 'easier or as hard'} for the model nobody tuned "
      "(a hunch: two readings on different questions, each with its own range, not a test of the difference).")
dev_vs_test.round(3)

**Finding or hunch, claim by claim.**

* **Finding:** each challenger's verdict against the bar, because the bar was written first and both paired tests decide
  it; the decision follows from those verdicts by the rule in section 1. A "does not clear" is a finding about the bar, not
  a finding that there is no difference (see *This is wrong if* above).
* **Hunch:** how much each gap shrank from dev to test, the champion's move from dev to test, and the weakest module,
  because each is one reading with no paired test, or rests on a handful of questions.

In [ ]:
def said(v):
    return "; ".join(f"{p}: {'pass' if ok else 'FAIL'}, {why}" for p, (ok, why) in v["parts"].items())


v_lines = " ".join(f"{c}: {'clears' if v['clears'] else 'does not clear'} ({said(v)})." for c, v in verdicts.items())
paired_text = "; ".join(f"{c} on {s}: {x['only_second']} only the challenger found against {x['only_first']}, gap {x['gap_recall_at_5']:+.3f} "
                        f"[{x['gap_low']:+.3f}, {x['gap_high']:+.3f}], p {x['p_value']:.4f}" for (c, s), x in cmp.items())
speed_text = "; ".join(f"{n} {sp['ms_per_question_median']} ms (middle spread {sp['ms_middle_spread']}, full spread {sp['ms_spread']})" for n, sp in speed.items())
tables = "; ".join(f"{s.name}: honest {results[s.name]['sets']['honest']['summary']['recall_at_5']:.3f} "
                   f"[{results[s.name]['sets']['honest']['summary']['recall_at_5_low']:.2f}, {results[s.name]['sets']['honest']['summary']['recall_at_5_high']:.2f}], "
                   f"hit@1 {results[s.name]['sets']['honest']['summary']['hit_at_1']:.3f}, MRR {results[s.name]['sets']['honest']['summary']['mrr']:.3f}; "
                   f"originals {results[s.name]['sets']['originals']['summary']['recall_at_5']:.3f} "
                   f"[{results[s.name]['sets']['originals']['summary']['recall_at_5_low']:.2f}, {results[s.name]['sets']['originals']['summary']['recall_at_5_high']:.2f}], "
                   f"hit@1 {results[s.name]['sets']['originals']['summary']['hit_at_1']:.3f}; all {results[s.name]['sets']['all']['summary']['recall_at_5']:.3f}; "
                   f"weakest module with 5+ questions {weakest[s.name]}; {speed[s.name]['ms_per_question_median']} ms a question (middle spread {speed[s.name]['ms_middle_spread']}), "
                   f"{sizes[s.name]} MB" for s in CONTENDERS)
curse = "; ".join(f"{s.name} gap over the champion {dev_vs_test.loc[s.name, 'dev gap over champion']:+.3f} on dev, "
                  f"{dev_vs_test.loc[s.name, f'{SPLIT} gap over champion']:+.3f} on {SPLIT}" for s in CHALLENGERS)
description = f"""**Stage 4 of the coach-retrieval lab: {'a DRY RUN on dev, checking the code, not a result' if DRY_RUN else 'the test split, opened once'}.** Built by `04_evaluate.ipynb`; one child run per contender holds its numbers, ranked rows and the {SPLIT} datasets as inputs (pairs_{SPLIT} {ds_pairs.digest}, honest_{SPLIT} {ds_honest.digest}, corpus {ds_corpus.digest}; stage-1 run {tracking.short(stage1.info.run_id)}, stage 2 {tracking.short(stage2.info.run_id)}, stage 3 {tracking.short(stage3.info.run_id)}).

**What.** Three contenders on the {len(lessons_eval)} {SPLIT} lessons: the champion {r.CHAMPION.name}; challenger (a) the fine-tuned candidate {CANDIDATE.name} (run {tracking.short(cand_run.info.run_id)}, tagged candidate=stage4), loaded from its logged MLflow model {cand_model.model_id[:10]}, not a folder (identity check: it gives {identity['honest']:.3f} on dev honest, its run logged {dev[CANDIDATE.name]['honest']:.3f}); challenger (b) {r.GEMMA.name} as shipped. Question sets: honest_{SPLIT} ({honest_state}), the same {n_o} questions as the course wrote them (originals), and all {len(pairs_eval)} {SPLIT} pairs by group (option chips outside the bar). Whole corpus of {len(corpus)} paragraphs searched.

**Why.** {'A dry run of the once-only look, on dev, so a code error cannot crash the real one half-way; the bar and decision below are the code checking itself, not findings. The real look is' if DRY_RUN else 'The once-only look at questions nothing was chosen on,'} held to a bar written before it: does either challenger find more lessons for learner-worded questions than the champion, without forgetting the course-worded ones, at no worse speed?

**The bar (pre-registered, written {written_at} UTC, {SPLIT} opened {opened_at} UTC, digest {bar_digest}; the metrics bar_logged and {OPENED_METRIC} carry millisecond timestamps that show the order).** {the_bar}

**Numbers.** {tables}. Paired against the champion: {paired_text}. Chance of at least one fluke: {fwer_wins:.3f} for the two win tests at {P_EACH:g} ({fwer_unshared:.3f} had each used 0.05; {fwer_all:.3f} for all {looks} paired tests at 0.05). Speed (median of 5 timed passes on {n_o} questions, batch 32, after one untimed warm-up pass each, the order turned each round; {'read back from the first look' if timed_by is not None else 'timed in this pass'}): {speed_text}.

**The bar, applied.** {v_lines}

**Decision.** {decision} {f'Tagged {WINNER_TAG}=true on child run {tracking.short(winner_tagged_on)}.' if winner_tagged_on else f'The same decision as the first look, run {tracking.short(decided_by.info.run_id)}, which holds the {WINNER_TAG} tag; none is set here.'}

**Dev against {SPLIT} (selection bias, the winner's curse).** {curse}; the champion moved {dev_vs_test.loc[r.CHAMPION.name, 'move']:+.3f}.

**Wrong if.** The honest test is model-made ({r.PARAPHRASE_MODEL}) and model-read (Claude, at the owner's request); the candidate's training rewrites share its writer and prompt, so it may have learned the rewriter's style; {len(lessons_eval)} {SPLIT} lessons and {n_h} learner-worded questions are few (Wilson ranges about 18 points wide; the margin lines say how many questions each verdict rests on), and "not shown" is not "no difference"; dev was used for choosing; scored on stage 1's frozen snapshot of the course (fingerprint {fingerprint_stage1}; today {fingerprint_now}, {changed_wording} {SPLIT} questions worded differently).

**The rule from here.** Re-running this notebook is the same look again (same frozen models, same questions, same bar, the first look's own timing read back). Nothing may be retrained, re-chosen or have its bar changed after this run; the notebook refuses a pass whose candidate, bar or decision differs, and any pass that logged test_opened_at counts as a look, whatever its status. The holdout stays locked until stage 5."""
mlflow.set_tag("mlflow.note.content", description)
print(description)

## 6. What stage 5 does next, and why the holdout is still locked (*Model registry and promotion*)

**Stage 5 registers the winner.** `05_register.ipynb` takes the run tagged `stage4_winner=true`, registers its logged model in
the local MLflow **model registry** (the shelf of named, versioned models; glossary: model registry) and gives it an
**alias**, a movable name such as `champion` or `challenger` that the app loads by, so swapping models is moving a name, not
changing code (glossary: alias). If the champion stayed, stage 5 registers bge-large as shipped under `champion` and no
alias moves to anything new.

**Why the holdout is still locked.** The 18 holdout lessons have never been scored by anything. Test has now been looked at
once, and the decision above was made on it, so test is no longer clean for the *next* decision. The holdout is the last
once-only check, scored at stage 5 for the winner alone just before its alias moves: a final confirmation on questions no
choice of any kind has touched. `retrieval.score` still refuses it without `yes_holdout=True`.

**And the rule, once more.** From the moment this notebook has run on test, the candidate is frozen: no retraining, no
re-picking a run or a checkpoint, no new bar. A better idea (more rewrites, a lower skip ratio, a longer LoRA run) is a
new challenger for a later round, judged on new questions (stage 7's typed ones, or a fresh split dealt on purpose), never
on this test set again.

In [ ]:
mlflow.log_metric("minutes_all", round((time.monotonic() - T0) / 60, 2))
mlflow.end_run()
print(f"This pass is saved: {tracking.run_link(parent)}")
for name, child in child_runs.items():
    print(f"  {name:45} {tracking.short(child)}" + ("  <- " + WINNER_TAG if winner_tagged_on and name == winner.name else ""))